## A multi-agent AI workflow that automatically summarizes, classifies, and recommends actions for customer feedback.

### Problem Statement

Customer feedback is a critical source of insight for improving products and services. However, organizations often receive large volumes of unstructured feedback from support tickets, surveys, app reviews, and emails. Manually reading, summarizing, categorizing, and deciding actions for each piece of feedback is time-consuming, inconsistent, and difficult to scale.

To address this challenge, we build a multi-agent AI workflow using the Agent Framework and Azure AI services. The solution uses a sequence of specialized AI agents that collaboratively process feedback:

1. **Summarizer Agent** – Converts long customer feedback into a concise summary.

2. **Classifier Agent** – Categorizes the feedback as Positive, Negative, or Feature Request.

3. **Action Agent** – Recommends the next best action for the business team.

By orchestrating these agents in a sequential workflow, organizations can automatically analyze customer feedback, prioritize issues, and generate actionable insights for product, support, and engineering teams.

<img src="./image.png" alt="images" width="350" style="display:block; margin-top:10px;"/>

In [47]:
%pip install python-dotenv azure-identity agent-framework==1.0.0rc2 opentelemetry-semantic-conventions-ai==0.4.13

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\vijay\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [48]:
import asyncio
from typing import cast

from agent_framework import Message
from agent_framework.azure import AzureAIAgentClient
from agent_framework.orchestrations import SequentialBuilder

from azure.identity import AzureCliCredential
from dotenv import load_dotenv

In [49]:
load_dotenv()
print("Environment variables loaded")

Environment variables loaded


In [50]:
summarizer_instructions = """
Summarize the customer's feedback in one short sentence.
Keep it neutral and concise.

Example output:
App crashes during photo upload.
User praises dark mode feature.
"""

classifier_instructions = """
Classify the feedback as one of the following:
Positive, Negative, or Feature request.
"""

action_instructions = """
Based on the summary and classification,
suggest the next action in one short sentence.

Example output:
Escalate as a high-priority bug for the mobile team.
Log as positive feedback to share with design and marketing.
Log as enhancement request for product backlog.
"""

In [51]:
feedback = """
I use the dashboard every day to monitor metrics, and it works well overall. 
But when I'm working late at night, the bright screen is really harsh on my eyes. 
If you added a dark mode option, it would make the experience much more comfortable.
"""

print(feedback)


I use the dashboard every day to monitor metrics, and it works well overall. 
But when I'm working late at night, the bright screen is really harsh on my eyes. 
If you added a dark mode option, it would make the experience much more comfortable.



In [52]:
async def run_workflow(feedback):

    credential = AzureCliCredential()

    async with AzureAIAgentClient(credential=credential) as chat_client:

        # Create agents
        summarizer = chat_client.as_agent(
            instructions=summarizer_instructions,
            name="summarizer",
        )

        classifier = chat_client.as_agent(
            instructions=classifier_instructions,
            name="classifier",
        )

        action = chat_client.as_agent(
            instructions=action_instructions,
            name="action",
        )

        # Build sequential workflow
        workflow = SequentialBuilder(
            participants=[summarizer, classifier, action]
        ).build()

        workflow_input = f"Customer feedback:\n{feedback}"
        outputs: list[list[Message]] = []

        response_stream = workflow.run(workflow_input, stream=True)
        async for update in response_stream:
            if update.type == "output":
                outputs.append(cast(list[Message], update.data))

        # Get complete response (if this SDK stream supports it)
        if hasattr(response_stream, "get_final_response") and callable(getattr(response_stream, "get_final_response")):
            final = await response_stream.get_final_response()
            if getattr(final, "messages", None):
                outputs.append(cast(list[Message], final.messages))

        # Display outputs
        if outputs:
            for i, msg in enumerate(outputs[-1], start=1):
                name = msg.author_name or (
                    "assistant" if msg.role == "assistant" else "user"
                )
                print(f"{'-' * 60}\n{i:02d} [{name}]\n{msg.text}")

In [53]:
await run_workflow(feedback)

------------------------------------------------------------
01 [user]
Customer feedback:

I use the dashboard every day to monitor metrics, and it works well overall. 
But when I'm working late at night, the bright screen is really harsh on my eyes. 
If you added a dark mode option, it would make the experience much more comfortable.

------------------------------------------------------------
02 [summarizer]
Request for a dark mode option for better nighttime usability.
------------------------------------------------------------
03 [classifier]
Feature request.
------------------------------------------------------------
04 [action]
Log as enhancement request for product backlog.
